#  Predicción de Diabetes con K-Nearest Neighbors (KNN)

**Universidad Politécnica Salesiana**  
**Carrera:** Computación  
**Asignatura:** Inteligencia Artificial  
**Tema:** Implementación del algoritmo K-Nearest Neighbors (KNN) para la predicción de diabetes  
**Estudiantes:** José Vanegas y Miguel Vanegas  
**Docente:** Ing. Remigio Hurtado  
**Fecha:** 18 de mayo de 2026  
**Ciudad:** Cuenca – Ecuador


#  Índice

1. Introducción
2. Objetivos
3. Carga y exploración del dataset
4. Identificación de variables
5. Transformación y preprocesamiento
6. Fundamento del algoritmo KNN
7. Implementación manual del KNN
8. Predicción de nuevos pacientes
9. Conclusiones


#  Introducción

La diabetes es una enfermedad crónica que puede detectarse mediante el análisis de variables demográficas y clínicas. En esta práctica se utiliza el algoritmo **K-Nearest Neighbors (KNN)** para clasificar si un paciente presenta o no diabetes.

El enfoque consiste en:
- Cargar un conjunto de datos en formato CSV.
- Identificar los tipos de variables.
- Transformar y estandarizar los datos.
- Implementar manualmente el cálculo de distancias.
- Predecir el diagnóstico de un nuevo paciente.

Esta práctica permite comprender de manera práctica cómo funciona KNN y por qué el preprocesamiento es esencial para obtener resultados confiables.


#  Objetivos

## Objetivo General
Aplicar el algoritmo K-Nearest Neighbors (KNN) para predecir si un paciente tiene diabetes a partir de variables demográficas y de salud.

## Objetivos Específicos
1. Cargar y explorar el dataset de pacientes.
2. Identificar variables numéricas, nominales, ordinales y la variable objetivo.
3. Transformar las variables categóricas a formato numérico.
4. Estandarizar los datos para el cálculo de distancias.
5. Implementar manualmente el algoritmo KNN.
6. Realizar predicciones sobre nuevos pacientes.
7. Interpretar los resultados obtenidos.


#  3. Carga y exploración del dataset

En esta sección se importan las librerías necesarias y se carga el archivo CSV que contiene los datos de los pacientes. Luego se crea una copia del dataset original para trabajar sin modificar la información inicial.

**Resultado esperado:** se muestra el tamaño del dataset y las primeras filas para verificar que los datos se cargaron correctamente.

In [2]:

import numpy as np
import pandas as pd
import copy
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


ruta = "data.csv"
dfOriginal = pd.read_csv(ruta)
cData = copy.deepcopy(dfOriginal)
print("Tamaño del dataset:", cData.shape)


print(cData.head(10))

Tamaño del dataset: (6, 5)
   sexo     ciudad colesterol  edad diabetes
0     1     Cuenca       bajo    18       no
1     2      Quito       alto    52       si
2     2  Guayaquil      medio    34       no
3     1       Loja       alto    61       si
4     2     Ambato      medio    45       no
5     1    Machala   muy alto    67       si


#  4. Identificación de variables

Aquí se clasifican las variables del dataset según su tipo. Esta clasificación es importante porque cada tipo de variable necesita un tratamiento diferente antes de aplicar KNN.

- **Variables numéricas:** se pueden usar directamente, pero normalmente se estandarizan.
- **Variables nominales:** no tienen orden, por eso se convierten con One Hot Encoding.
- **Variables ordinales:** sí tienen orden, por eso se convierten respetando su jerarquía.
- **Variable objetivo:** es la columna que se desea predecir.

In [7]:

var_numericas = ["edad"]

var_nominales = ["ciudad", "sexo"]

var_ordinales = ["colesterol"]

var_objetivo = "diabetes"

X = cData.drop(columns=["diabetes"])
y = cData["diabetes"].map({"no": 0, "si": 1})

print("Variables numéricas:", var_numericas)
print("Variables categóricas nominales:", var_nominales)
print("Variables categóricas ordinales:", var_ordinales)
print("Variable objetivo:", var_objetivo)
print("Variable objetivo transformada:")
print(y.values)

Variables numéricas: ['edad']
Variables categóricas nominales: ['ciudad', 'sexo']
Variables categóricas ordinales: ['colesterol']
Variable objetivo: diabetes
Variable objetivo transformada:
[0 1 0 1 0 1]


#  5. Transformación y preprocesamiento

Antes de usar KNN, los datos deben transformarse a valores numéricos porque el algoritmo calcula distancias matemáticas.

En esta sección se realiza lo siguiente:

1. Se define el orden de la variable **colesterol**.
2. Se aplica **One Hot Encoding** a las variables nominales.
3. Se aplica **Ordinal Encoding** a la variable ordinal.
4. Se estandarizan todas las variables con **StandardScaler**.
5. Se reconstruye una tabla final para visualizar los datos transformados.

La estandarización es importante porque KNN trabaja con distancias. Si una variable tiene valores muy grandes, podría influir demasiado en el resultado.

In [11]:
# Orden lógico de la variable colesterol
orden_var = [["bajo", "medio", "alto", "muy alto"]]

preprocesador = ColumnTransformer(transformers=[
    ('cat_nom', OneHotEncoder(sparse_output=False, handle_unknown="ignore"), var_nominales),

    ('cat_ord', OrdinalEncoder(categories=orden_var), var_ordinales)
], remainder='passthrough')

pipe_maestro = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('estandarizacion_total', StandardScaler())
])

X_transformado = pipe_maestro.fit_transform(X)
nombres_nominales = pipe_maestro.named_steps['preprocesamiento']     .named_transformers_['cat_nom']     .get_feature_names_out(var_nominales)
columnas_finales = list(nombres_nominales) + var_ordinales + var_numericas
df_final = pd.DataFrame(data=X_transformado, columns=columnas_finales)
print("********** TABLA ESTANDARIZADA **********")
print(df_final.round(4).head(6).to_string(index=False))

********** TABLA ESTANDARIZADA **********
 ciudad_Ambato  ciudad_Cuenca  ciudad_Guayaquil  ciudad_Loja  ciudad_Machala  ciudad_Quito  sexo_1  sexo_2  colesterol    edad
       -0.4472         2.2361           -0.4472      -0.4472         -0.4472       -0.4472     1.0    -1.0     -1.5667 -1.7085
       -0.4472        -0.4472           -0.4472      -0.4472         -0.4472        2.2361    -1.0     1.0      0.5222  0.3538
       -0.4472        -0.4472            2.2361      -0.4472         -0.4472       -0.4472    -1.0     1.0     -0.5222 -0.7380
       -0.4472        -0.4472           -0.4472       2.2361         -0.4472       -0.4472     1.0    -1.0      0.5222  0.8997
        2.2361        -0.4472           -0.4472      -0.4472         -0.4472       -0.4472    -1.0     1.0     -0.5222 -0.0708
       -0.4472        -0.4472           -0.4472      -0.4472          2.2361       -0.4472     1.0    -1.0      1.5667  1.2637


#  6. Fundamento del algoritmo KNN

KNN significa **K-Nearest Neighbors** o **K vecinos más cercanos**. El algoritmo compara un nuevo paciente con los pacientes ya existentes y busca los más parecidos.

En esta práctica se utiliza la **distancia euclidiana**, que mide qué tan lejos está un paciente de otro según sus características transformadas.

La idea principal es:

1. Transformar el nuevo paciente con el mismo pipeline.
2. Calcular la distancia entre el nuevo paciente y cada paciente del dataset.
3. Ordenar las distancias de menor a mayor.
4. Tomar los **k vecinos más cercanos**.
5. Hacer una votación para decidir si el paciente tiene diabetes o no.

#  7. Implementación manual del algoritmo KNN

En esta sección se implementa KNN manualmente. Esto permite entender cómo funciona el algoritmo por dentro, sin usar directamente una clase como `KNeighborsClassifier`.

El código se divide en tres partes:

- Preparación de datos para KNN.
- Función para calcular distancia euclidiana.
- Función para predecir usando los vecinos más cercanos.

In [14]:
# Datos finales que usará el algoritmo KNN
X_knn = X_transformado

# Etiquetas reales: 0 significa no diabetes, 1 significa sí diabetes
y_knn = y.values

# Función para calcular la distancia euclidiana entre dos pacientes
def distancia_euclidea(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

# Función principal para aplicar KNN manualmente
def knn_predict(nuevo_paciente, k=3):

    # 1. Convertir el nuevo paciente a DataFrame
    # Esto es necesario porque el pipeline espera datos en formato tabla
    nuevo_df = pd.DataFrame([nuevo_paciente])

    # 2. Transformar el nuevo paciente usando el mismo pipeline del entrenamiento
    # Así se asegura que tenga el mismo formato que los datos originales
    nuevo_transformado = pipe_maestro.transform(nuevo_df)

    # 3. Calcular distancias entre el nuevo paciente y cada paciente del dataset
    distancias = []

    for i in range(len(X_knn)):
        dist = distancia_euclidea(nuevo_transformado[0], X_knn[i])
        distancias.append((i, dist, y_knn[i]))

    # 4. Ordenar las distancias de menor a mayor
    # La posición 1 de la tupla representa la distancia
    distancias.sort(key=lambda x: x[1])

    # 5. Seleccionar los k vecinos más cercanos
    vecinos = distancias[:k]

    # 6. Mostrar todas las distancias ordenadas
    print("--- DISTANCIAS ORDENADAS ---")
    for i, d, label in distancias:
        print(f"Paciente {i} → Distancia: {round(d, 4)} → Clase real: {label}")

    # 7. Mostrar solo los vecinos más cercanos
    print(f"--- VECINOS MÁS CERCANOS (k={k}) ---")
    for i, d, label in vecinos:
        print(f"Paciente {i} → Distancia: {round(d, 4)} → Clase real: {label}")

    # 8. Realizar votación entre los vecinos
    votos = [label for _, _, label in vecinos]
    prediccion = max(set(votos), key=votos.count)

    # 9. Mostrar resultado final
    print("--- PREDICCIÓN FINAL ---")
    print("Diabetes:", "SÍ" if prediccion == 1 else "NO")

    return prediccion

#  8. Predicción de nuevos pacientes

Esta sección permite ingresar manualmente los datos de un nuevo paciente. Luego, esos datos se envían a la función `knn_predict()` para obtener la predicción.

Los datos que se solicitan son:

- Sexo
- Ciudad
- Nivel de colesterol
- Edad

Después de ingresar los datos, el programa muestra las distancias, los vecinos más cercanos y la predicción final.

In [16]:
# Nuevo paciente de prueba
nuevo = {
    "sexo": 2,
    "ciudad": "Cuenca",
    "colesterol": "alto",
    "edad": 50
}

# Ingresar paciente y realizar predicción con k = 3

knn_predict(nuevo, k=3)

--- DISTANCIAS ORDENADAS ---
Paciente 1 → Distancia: 3.7967 → Clase real: 1
Paciente 4 → Distancia: 3.9475 → Clase real: 0
Paciente 0 → Distancia: 4.0163 → Clase real: 0
Paciente 2 → Distancia: 4.0537 → Clase real: 0
Paciente 3 → Distancia: 4.7797 → Clase real: 1
Paciente 5 → Distancia: 4.9552 → Clase real: 1
--- VECINOS MÁS CERCANOS (k=3) ---
Paciente 1 → Distancia: 3.7967 → Clase real: 1
Paciente 4 → Distancia: 3.9475 → Clase real: 0
Paciente 0 → Distancia: 4.0163 → Clase real: 0
--- PREDICCIÓN FINAL ---
Diabetes: NO


np.int64(0)

#  9. Conclusiones
En conclusión, el algoritmo KNN permite clasificar nuevos pacientes comparándolos con casos similares del dataset; la transformación de variables categóricas a valores numéricos y la estandarización son pasos esenciales para obtener distancias equilibradas y evitar que una sola variable domine la medida de similitud. La implementación manual del algoritmo facilita comprender el cálculo de distancias, el ordenamiento de vecinos y el proceso de votación, aunque en conjuntos de datos grandes su uso puede implicar un mayor costo computacional que requiere estrategias de optimización.